# Subqueries

In [7]:
import psycopg
import pandas as pd
conn = psycopg.connect(
    host = 'localhost',
    port= 5432,
    dbname ='chinook',
    user = 'postgres',
    password = '231165'
)

cur = conn.cursor()

## Subqueries

### Scaler Subquery

In [10]:
# Find tracks priced above the average track price

result = pd.read_sql('''
SELECT
    name,
    unit_price
FROM track
WHERE unit_price > (
        SELECT AVG(unit_price)
        FROM track
)

''', conn)

result


C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_14688\2502409353.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,name,unit_price
0,Battlestar Galactica: The Story So Far,1.99
1,Occupation / Precipice,1.99
2,"Exodus, Pt. 1",1.99
3,"Exodus, Pt. 2",1.99
4,Collaborators,1.99
...,...,...
208,"There's No Place Like Home, Pt. 1",1.99
209,"There's No Place Like Home, Pt. 2",1.99
210,"There's No Place Like Home, Pt. 3",1.99
211,Branch Closing,1.99


### Subquery with IN

In [11]:
# Find tracks belonging to a selected set of albums

result = pd.read_sql('''
SELECT
    name,
    album_id
FROM track
WHERE album_id IN (
    SELECT album_id
    FROM album
    WHERE artist_id IN (1, 2)
)
ORDER BY album_id;

''', conn)

result


C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_14688\3901661235.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,name,album_id
0,For Those About To Rock (We Salute You),1
1,Put The Finger On You,1
2,Let's Get It Up,1
3,Inject The Venom,1
4,Snowballed,1
5,Evil Walks,1
6,C.O.D.,1
7,Breaking The Rules,1
8,Night Of The Long Knives,1
9,Spellbound,1


### Corelated Subquery

In [12]:
# Compare each track with its own album average

query = """
SELECT
    t.name,
    t.album_id,
    t.milliseconds
FROM track AS t
WHERE t.milliseconds > (
    SELECT AVG(t2.milliseconds)
    FROM track AS t2
    WHERE t2.album_id = t.album_id
)
ORDER BY t.album_id, t.milliseconds DESC;
"""

df = pd.read_sql_query(query, conn)
df.head(10)

C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_14688\3666672635.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,name,album_id,milliseconds
0,For Those About To Rock (We Salute You),1,343719
1,Spellbound,1,270863
2,Evil Walks,1,263497
3,Breaking The Rules,1,263288
4,Princess of the Dawn,3,375418
5,Overdose,4,369319
6,Let There Be Rock,4,366654
7,Go Down,4,331180
8,Problem Child,4,325041
9,Whole Lotta Rosie,4,323761


## EXIST & NOT EXIST

### EXIST

In [13]:
# Find customers who have at least one invoice

query = """
SELECT
    c.first_name,
    c.last_name
FROM customer AS c
WHERE EXISTS (
    SELECT 1
    FROM invoice AS i
    WHERE i.customer_id = c.customer_id
)
ORDER BY c.customer_id;
"""

df = pd.read_sql_query(query, conn)
df.head(10)

C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_14688\138070114.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,first_name,last_name
0,Luís,Gonçalves
1,Leonie,Köhler
2,François,Tremblay
3,Bjørn,Hansen
4,František,Wichterlová
5,Helena,Holý
6,Astrid,Gruber
7,Daan,Peeters
8,Kara,Nielsen
9,Eduardo,Martins


### NOT EXIST

In [14]:
# Find customers who have never purchased anything

query = """
SELECT
    c.customer_id,
    c.first_name,
    c.last_name
FROM customer AS c
WHERE NOT EXISTS (
    SELECT 1
    FROM invoice AS i
    WHERE i.customer_id = c.customer_id
)
ORDER BY c.customer_id;
"""

df = pd.read_sql_query(query, conn)
df

C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_14688\2501211676.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,customer_id,first_name,last_name
